In [9]:
import os
import re
import sqlite3
import pandas as pd

In [10]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DB_PATH = os.path.join(BASE_DIR, "2_DATABASE", "eko_cards.db")
INPUT_FILE = os.path.join(BASE_DIR, "3_PROJECT_DATA", "eko_transactions.xlsx")

OUTPUT_FOLDER = os.path.join(BASE_DIR, "VESELA")
OUTPUT_FILE = os.path.join(OUTPUT_FOLDER, "eko_transactions.xlsx")

GTA_EIK = "200871895"


def clean_product(value):
    if pd.isna(value):
        return ""

    return (
        str(value)
        .replace("\xa0", " ")
        .replace("\u00a0", " ")
        .replace("\t", " ")
        .strip()
        .upper()
    )


def normalize_transaction_product(value):
    if pd.isna(value):
        return ""

    value = (
        str(value)
        .replace("\xa0", " ")
        .replace("\u00a0", " ")
        .replace("\t", " ")
        .strip()
    )

    # Премахва продуктов код отпред, ако има такъв:
    # пример: "110000000 Diesel EKONOMY" -> "Diesel EKONOMY"
    value = re.sub(r"^\d{6,}\s*", "", value)

    # Нормализира интервалите
    value = re.sub(r"\s+", " ", value)

    return value.strip().upper()


def clean_eik(value):
    if pd.isna(value):
        return ""

    return re.sub(r"\D", "", str(value))


def add_price_to_gta_column():

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # -------------------------
    # READ ORIGINAL EKO FILE
    # -------------------------
    original_df = pd.read_excel(
        INPUT_FILE,
        dtype={
            "Number": str,
            "Billing Document": str
        }
    )

    original_df.columns = original_df.columns.str.strip()

    required_columns = ["Date", "Material", "FinPr"]

    missing_columns = [
        col for col in required_columns
        if col not in original_df.columns
    ]

    if missing_columns:
        raise ValueError(f"Missing required columns in Excel file: {missing_columns}")

    work_df = original_df.copy()
    work_df["_row_id"] = range(len(work_df))

    # -------------------------
    # PREPARE TRANSACTION DATA
    # -------------------------
    work_df["_transaction_date"] = pd.to_datetime(
        work_df["Date"],
        dayfirst=True,
        errors="coerce"
    )

    work_df["_product"] = work_df["Material"].apply(normalize_transaction_product)

    # -------------------------
    # READ GTA PRICES FROM DATABASE
    # -------------------------
    conn = sqlite3.connect(DB_PATH)

    prices_df = pd.read_sql(
        """
        SELECT
            rowid AS price_rowid,
            eik,
            product,
            date,
            final_price
        FROM prices
        """,
        conn
    )

    conn.close()

    # -------------------------
    # CLEAN PRICES
    # -------------------------
    prices_df["eik"] = prices_df["eik"].apply(clean_eik)
    prices_df["product"] = prices_df["product"].apply(clean_product)
    prices_df["date"] = pd.to_datetime(prices_df["date"], errors="coerce")

    prices_df["final_price"] = pd.to_numeric(
        prices_df["final_price"],
        errors="coerce"
    ).round(3)

    # Взимаме САМО цените за ДЖИ ТИ ЕЙ ПЕТРОЛИУМ ООД
    prices_df = prices_df[
        prices_df["eik"] == GTA_EIK
    ]

    prices_df = prices_df.dropna(
        subset=["product", "date", "final_price"]
    )

    # -------------------------
    # FIND VALID PRICE BY PERIOD
    # За всяка транзакция взима последната валидна цена:
    # prices.date <= transaction Date
    # -------------------------
    candidates_df = work_df[
        [
            "_row_id",
            "_transaction_date",
            "_product"
        ]
    ].merge(
        prices_df,
        left_on="_product",
        right_on="product",
        how="left"
    )

    candidates_df = candidates_df[
        candidates_df["date"] <= candidates_df["_transaction_date"]
    ]

    candidates_df = candidates_df.sort_values(
        by=["_row_id", "date", "price_rowid"]
    )

    latest_prices_df = candidates_df.drop_duplicates(
        subset=["_row_id"],
        keep="last"
    )[
        [
            "_row_id",
            "final_price"
        ]
    ].rename(columns={
        "final_price": "PriceToGTA"
    })

    # -------------------------
    # ADD PriceToGTA TO ORIGINAL FILE
    # -------------------------
    work_df = work_df.merge(
        latest_prices_df,
        on="_row_id",
        how="left"
    )

    result_df = original_df.copy()

    if "PriceToGTA" in result_df.columns:
        result_df = result_df.drop(columns=["PriceToGTA"])

    finpr_position = result_df.columns.get_loc("FinPr") + 1

    result_df.insert(
        finpr_position,
        "PriceToGTA",
        work_df["PriceToGTA"].round(3)
    )

    # -------------------------
    # EXPORT SINGLE FILE
    # -------------------------
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        result_df.to_excel(
            writer,
            index=False,
            sheet_name="Sheet1"
        )

        worksheet = writer.sheets["Sheet1"]

        price_col_idx = result_df.columns.get_loc("PriceToGTA") + 1

        for row in range(2, worksheet.max_row + 1):
            worksheet.cell(row=row, column=price_col_idx).number_format = "0.000"

        for column_cells in worksheet.columns:
            max_length = 0
            column_letter = column_cells[0].column_letter

            for cell in column_cells:
                if cell.value is not None:
                    max_length = max(max_length, len(str(cell.value)))

            worksheet.column_dimensions[column_letter].width = min(max_length + 2, 35)

    print("DONE")
    print(f"Input file: {INPUT_FILE}")
    print(f"Output file: {OUTPUT_FILE}")
    print(f"GTA EIK used: {GTA_EIK}")
    print(f"Rows in original file: {len(result_df)}")
    print(f"Rows with matched PriceToGTA: {result_df['PriceToGTA'].notna().sum()}")
    print(f"Rows without matched PriceToGTA: {result_df['PriceToGTA'].isna().sum()}")


add_price_to_gta_column()

DONE
Input file: C:\Users\sveto\OneDrive\Desktop\eko_inv\3_PROJECT_DATA\eko_transactions.xlsx
Output file: C:\Users\sveto\OneDrive\Desktop\eko_inv\VESELA\eko_transactions.xlsx
GTA EIK used: 200871895
Rows in original file: 1816
Rows with matched PriceToGTA: 1561
Rows without matched PriceToGTA: 255
